# Step 3：LLaMA-Factory QLoRA 微调训练
## 使用说明
1. 打开 Colab：https://colab.research.google.com
2. 菜单 → 修改 → 笔记本设置 → 硬件加速器选 **T4 GPU**
3. 上传你的 `filtered_dataset.json` 到左侧文件栏
4. 按顺序运行每个单元格即可

In [ ]:
# 单元格 1：克隆 LLaMA-Factory 并安装依赖（约 3 分钟）
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
!pip install -e '.[torch,bitsandbytes,metrics]' -q
!pip install bitsandbytes>=0.39.0

In [ ]:
# 单元格 2：把上传的数据集复制到 data 目录，并注册
import json, shutil

# 复制数据文件
shutil.copy('/content/filtered_dataset.json', 'data/filtered_dataset.json')

# 读取现有的 dataset_info.json 并添加自定义数据集
with open('data/dataset_info.json', 'r', encoding='utf-8') as f:
    info = json.load(f)

info['edu_difficulty'] = {
    'file_name': 'filtered_dataset.json',
    'columns': {
        'prompt': 'instruction',
        'query':  'input',
        'response': 'output'
    }
}

with open('data/dataset_info.json', 'w', encoding='utf-8') as f:
    json.dump(info, f, ensure_ascii=False, indent=2)

print('✅ 数据集注册完成，共', len(json.load(open('data/filtered_dataset.json'))), '条')

In [ ]:
# 单元格 3：写训练配置文件
config = """
### 模型
model_name_or_path: Qwen/Qwen2.5-0.5B-Instruct
trust_remote_code: true

### 量化（省显存，T4 必须开）
quantization_bit: 4
quantization_method: bnb

### 微调方式
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 16
lora_target: all

### 数据集
dataset: edu_difficulty
template: qwen
cutoff_len: 512
overwrite_cache: true
preprocessing_num_workers: 4

### 输出
output_dir: saves/qwen-edu/lora/sft
logging_steps: 5
save_steps: 100
plot_loss: true
overwrite_output_dir: true
report_to: none

### 训练参数
per_device_train_batch_size: 2
gradient_accumulation_steps: 4
learning_rate: 1.0e-4
num_train_epochs: 5.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
fp16: true

### 评估
val_size: 0.1
per_device_eval_batch_size: 2
eval_strategy: steps
eval_steps: 50
"""

with open('edu_train_config.yaml', 'w') as f:
    f.write(config.strip())
print('✅ 训练配置文件写入完成')

In [ ]:
# 单元格 4：开始训练（160条内数据集约 1-5 分钟）
llamafactory-cli train edu_train_config.yaml

In [ ]:
# 单元格 5：模型评测 （若评测于本地跑，请忽略）
import json
import torch
import random
import numpy as np
import matplotlib
matplotlib.rc('font', family='Noto Sans CJK JP')
matplotlib.rc('axes', unicode_minus=False)
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from collections import defaultdict

random.seed(42)

# ── 加载测试集（分层抽样，每类5条，共20条）──────────────────────────────
with open('data/filtered_dataset.json', 'r') as f:
    filtered_data = json.load(f)

with open('data/raw_dataset.json', 'r') as f:
    raw_data = json.load(f)

def make_test_set(data, per_class=5, seed=42):
    random.seed(seed)
    by_label = defaultdict(list)
    for d in data:
        by_label[d['output'].strip()].append(d)
    test = []
    for label in ['简单', '中等', '困难', '竞赛']:
        samples = by_label[label][:]
        random.shuffle(samples)
        test.extend(samples[:per_class])
    random.shuffle(test)
    return test

# 用 filtered 数据做测试集（两组都用同一个测试集才公平）
test_data = make_test_set(filtered_data)
true_labels = [d['output'].strip() for d in test_data]
print(f"测试集分布：{ {l: true_labels.count(l) for l in ['简单','中等','困难','竞赛']} }")

# ── 推理函数 ──────────────────────────────────────────────────────────────
def load_model(adapter_path):
    model_path = 'Qwen/Qwen2.5-0.5B-Instruct'
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    base_model = AutoModelForCausalLM.from_pretrained(
        model_path, trust_remote_code=True,
        torch_dtype=torch.float16, device_map='auto'
    )
    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()
    return model, tokenizer

def predict(model, tokenizer, test_data):
    preds = []
    for item in test_data:
        prompt = f"判断下面这道题目的难度等级，从【简单、中等、困难、竞赛】中选一个输出。\n{item['input']}"
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([text], return_tensors='pt').to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False)
        pred = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        label = '中等'
        for l in ['简单', '中等', '困难', '竞赛']:
            if l in pred:
                label = l
                break
        preds.append(label)
    return preds

# ── 跑两组评测 ────────────────────────────────────────────────────────────
LABELS = ['简单', '中等', '困难', '竞赛']
results = {}

# 组1：全量数据基线（需要你先训练好 full_baseline）
print("\n[1/2] 评测全量数据基线...")
model, tokenizer = load_model('saves/qwen-edu/lora/full_baseline')
preds_full = predict(model, tokenizer, test_data)
results['全量数据基线'] = preds_full
print(f"准确率：{accuracy_score(true_labels, preds_full)*100:.1f}%")
print(f"Macro F1：{f1_score(true_labels, preds_full, labels=LABELS, average='macro', zero_division=0)*100:.1f}%")
del model  # 释放显存

# 组2：K-means筛选 + rank=16
print("\n[2/2] 评测K-means筛选+rank=16...")
model, tokenizer = load_model('saves/qwen-edu/lora/rank16')
preds_kmeans = predict(model, tokenizer, test_data)
results['K-means+rank16'] = preds_kmeans
print(f"准确率：{accuracy_score(true_labels, preds_kmeans)*100:.1f}%")
print(f"Macro F1：{f1_score(true_labels, preds_kmeans, labels=LABELS, average='macro', zero_division=0)*100:.1f}%")
del model

# ── 生成对比图 ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 图1：准确率和F1对比柱状图
ax = axes[0]
names = list(results.keys())
accs = [accuracy_score(true_labels, results[n])*100 for n in names]
f1s  = [f1_score(true_labels, results[n], labels=LABELS, average='macro', zero_division=0)*100 for n in names]
x = np.arange(len(names))
bars1 = ax.bar(x - 0.2, accs, 0.35, label='准确率', color='#534AB7')
bars2 = ax.bar(x + 0.2, f1s,  0.35, label='Macro F1', color='#0F6E56')
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=10)
ax.set_ylim(0, 100)
ax.set_ylabel('分数 (%)')
ax.set_title('全量数据基线 vs K-means筛选')
ax.legend()
for bar in bars1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            f'{bar.get_height():.1f}%', ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            f'{bar.get_height():.1f}%', ha='center', fontsize=9)
ax.grid(axis='y', alpha=0.3)

# 图2：基线混淆矩阵
ax = axes[1]
cm1 = confusion_matrix(true_labels, results['全量数据基线'], labels=LABELS)
im = ax.imshow(cm1, cmap='Blues')
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(LABELS); ax.set_yticklabels(LABELS)
ax.set_xlabel('预测'); ax.set_ylabel('真实')
ax.set_title('混淆矩阵：全量数据基线')
for i in range(4):
    for j in range(4):
        ax.text(j, i, cm1[i,j], ha='center', va='center',
                color='white' if cm1[i,j] > cm1.max()/2 else 'black')

# 图3：K-means混淆矩阵
ax = axes[2]
cm2 = confusion_matrix(true_labels, results['K-means+rank16'], labels=LABELS)
im = ax.imshow(cm2, cmap='Greens')
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(LABELS); ax.set_yticklabels(LABELS)
ax.set_xlabel('预测'); ax.set_ylabel('真实')
ax.set_title('混淆矩阵：K-means+rank16')
for i in range(4):
    for j in range(4):
        ax.text(j, i, cm2[i,j], ha='center', va='center',
                color='white' if cm2[i,j] > cm2.max()/2 else 'black')

plt.tight_layout()
plt.savefig('saves/qwen-edu/lora/rank16/evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n 图片已保存为 evaluation_results.png')
print('左侧文件面板右键下载即可')

In [ ]:
# 单元格 6：训练完成后，打包权重文件准备下载
import shutil
shutil.make_archive('/content/lora_weights', 'zip', 'saves/qwen-edu/lora/sft')
print(' 权重已打包为 lora_weights.zip，请在左侧文件栏下载到本地')